# Kalam (كلام) : sous-titres français de la parole arabe, sur Colab

Ce notebook lance le serveur Kalam sur un GPU T4 et le rend accessible par un lien public HTTPS (tunnel Cloudflare, sans compte). Le lien s'ouvre sur un ordinateur ou sur un téléphone ; le HTTPS permet d'utiliser le micro.

**Avant de lancer**

1. Exécution > Modifier le type d'exécution > **GPU T4**.
2. Secrets Colab (icône de clé, à gauche) :
   - `HF_TOKEN` : jeton Hugging Face en lecture d'un compte qui a accepté les conditions du modèle [`CohereLabs/cohere-transcribe-arabic-07-2026`](https://huggingface.co/CohereLabs/cohere-transcribe-arabic-07-2026). Activer « Accès au notebook ».
   - `SCALEDOWN_API_KEY` (facultatif) : résumé de fin de session par l'API ScaleDown. Sans cette clé, le résumé est calculé en local.
3. Exécution > **Tout exécuter**. La première exécution est la plus longue (installation des paquets, téléchargement des modèles).

Sans `HF_TOKEN`, ou si l'accès au modèle Cohere n'est pas accordé, le notebook continue sans S3 : le direct (`/live`) est indisponible, mais la page `/video` fonctionne avec les moteurs Seamless et S0.

Guide complet : `docs/GUIDE.md` du dépôt.

## 1. Vérifier le GPU et installer les dépendances

In [ ]:
!nvidia-smi -L
!apt-get -qq install -y ffmpeg > /dev/null
# Combinaison vérifiée sur T4 (Python 3.13, torch 2.11 cu128, transformers 5.20, pyannote.audio 4.0.7, speechbrain 1.1, bitsandbytes 0.50, piper-tts 1.8).
# numpy<2.3 : sinon numba (librosa) refuse de s'importer et Seamless/S3 ne lisent plus l'audio. pyannote en dernier : il peut toucher à torch.
!pip -q install fastapi "uvicorn[standard]" python-multipart "gradio>=6.0" "transformers>=5.4.0" accelerate peft bitsandbytes sentencepiece protobuf librosa soundfile pyyaml sacremoses trafilatura "brotli>=1.2" yt-dlp "qrcode[pil]" requests python-docx "numpy<2.3"
!pip -q install "speechbrain>=1.0" "piper-tts>=1.2" "edge-tts>=6.1"
!pip -q install "pyannote.audio>=4.0" "numpy<2.3"
!python -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__, '| GPU', torch.cuda.is_available())"

## 2. Cloner le dépôt (branche `integration`)

In [ ]:
import os, subprocess, sys
DEPOT = '/content/NLP'
if os.path.isdir(DEPOT):
    subprocess.run(['git', '-C', DEPOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', 'integration', 'https://github.com/SamiNakibETU/NLP', DEPOT], check=True)
APP = DEPOT + '/speech_app'
print('branche :', subprocess.run(['git', '-C', DEPOT, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip(),
      '| commit :', subprocess.run(['git', '-C', DEPOT, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
print('projecteur S3 présent :', os.path.isfile(DEPOT + '/modeles/s3_temps1_50k/projecteur_meilleur.pt'))

## 3. Lire les secrets et choisir les moteurs

Les secrets ne sont jamais écrits dans le notebook. Cette cellule vérifie aussi que le jeton donne accès au modèle Cohere ; sinon S3 et le direct sont désactivés.

In [ ]:
from google.colab import userdata

def secret(nom):
    """Valeur du secret Colab `nom`, ou None s'il manque ou si l'accès au notebook n'est pas activé."""
    try:
        return (userdata.get(nom) or '').strip() or None
    except Exception:
        return None

for nom in ('HF_TOKEN', 'SCALEDOWN_API_KEY'):
    valeur = secret(nom)
    if valeur:
        os.environ[nom] = valeur
    else:
        os.environ.pop(nom, None)

COHERE = 'CohereLabs/cohere-transcribe-arabic-07-2026'
S3_OK = False
if not os.environ.get('HF_TOKEN'):
    print('HF_TOKEN absent : S3 et le direct sont désactivés. Ajouter le secret HF_TOKEN puis relancer cette cellule.')
else:
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(COHERE, 'config.json', token=os.environ['HF_TOKEN'])
        S3_OK = True
        print('Accès au modèle Cohere : oui')
    except Exception as exc:
        print(f'Accès au modèle Cohere refusé ({type(exc).__name__}). Accepter les conditions sur '
              f'https://huggingface.co/{COHERE} avec le compte du jeton, puis relancer cette cellule.')
        print('S3 et le direct sont désactivés ; /video reste disponible avec Seamless et S0.')

os.environ['PORTEPAROLE_NLP_REPO'] = DEPOT
os.environ['PORTEPAROLE_ENGINES'] = 's3,seamless,s0' if S3_OK else 'seamless,s0'   # moteurs de /video et /api/translate
os.environ['PORTEPAROLE_MT'] = 'off'                        # pas de traduction d'articles de presse (ancienne interface)
os.environ['PORTEPAROLE_SESSIONS'] = '/content/sessions'    # audio et segments de chaque session, relus par le rapport
os.environ['PORTEPAROLE_API_TOKEN'] = ''                    # mettre un jeton pour protéger le serveur (lien à compléter par ?token=...)
os.environ['PYTHONIOENCODING'] = 'utf-8'
LORA = DEPOT + '/modeles/lora_traduction'                   # adaptateur LoRA facultatif (utilisé seulement si PORTEPAROLE_TRAD_BLANC=qwen)
if os.path.isfile(LORA + '/adapter_config.json'):
    os.environ['PORTEPAROLE_LORA'] = LORA
else:
    os.environ.pop('PORTEPAROLE_LORA', None)
print('Moteurs de /video :', os.environ['PORTEPAROLE_ENGINES'])
print('Résumé de fin :', 'API ScaleDown' if os.environ.get('SCALEDOWN_API_KEY') else 'local (pas de clé SCALEDOWN_API_KEY)')

## 4. Lancer le serveur

In [ ]:
import re, time, requests
PORT = 7860
LOCAL = f'http://localhost:{PORT}'

def serveur_actif():
    try:
        return requests.get(LOCAL + '/api/health', timeout=2).ok
    except requests.RequestException:
        return False

def lancer_serveur():
    """Démarre uvicorn en arrière-plan (journal : /content/server.log) et attend qu'il réponde."""
    global serveur
    serveur = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api:app', '--host', '0.0.0.0', '--port', str(PORT)],
                               cwd=APP, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(180):
        if serveur_actif():
            return
        if serveur.poll() is not None:
            break
        time.sleep(1)
    raise RuntimeError('le serveur ne répond pas : lire le journal (dernière cellule)')

def arreter_serveur():
    if 'serveur' in globals() and serveur.poll() is None:
        serveur.terminate()
        try:
            serveur.wait(15)
        except subprocess.TimeoutExpired:
            serveur.kill()
    subprocess.run(['pkill', '-f', f'uvicorn api:app --host 0.0.0.0 --port {PORT}'])
    for _ in range(30):
        if not serveur_actif():
            return
        time.sleep(1)

if serveur_actif():
    print('Serveur déjà lancé sur le port', PORT)
else:
    lancer_serveur()
    print('Serveur prêt sur le port', PORT)

## 5. Ouvrir le tunnel public

Si le lien cesse de répondre, relancer cette cellule seule : un nouveau lien est créé (il change à chaque fois).

In [ ]:
if not os.path.isfile('/content/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.kill()
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', LOCAL],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for ligne in tunnel.stdout:
    if m := re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', ligne):
        URL = m.group(0)
        break
if not URL:
    raise RuntimeError('cloudflared n\'a pas donné de lien : relancer cette cellule')
print('Lien public :', URL)

## 6. Précharger les modèles du direct

S3, Opus-MT et NLLB-200 1,3B sont chargés avant la démonstration (environ une minute une fois les modèles téléchargés). Cellule sautée si S3 est désactivé.

In [ ]:
def precharger_direct(delai_s=900):
    requests.post(LOCAL + '/api/live/precharger').raise_for_status()
    debut = time.time()
    while time.time() - debut < delai_s:
        statut = requests.get(LOCAL + '/api/live/statut').json()
        if statut['groupe_charge'] == 'direct':
            print('Direct prêt en %.0f s | ffmpeg : %s | yt-dlp : %s' % (time.time() - debut, statut['ffmpeg'], statut['yt_dlp']))
            return True
        if 'Exception in thread precharge-live' in open('/content/server.log', encoding='utf-8', errors='replace').read():
            print('Le chargement du direct a échoué : lire le journal (dernière cellule).')
            return False
        time.sleep(3)
    print('Chargement trop long : lire le journal (dernière cellule).')
    return False

if S3_OK:
    precharger_direct()
else:
    print('S3 désactivé (voir la cellule 3) : pas de direct. La page /video fonctionne avec Seamless et S0.')

## 7. Lien et QR code

In [ ]:
import qrcode
from IPython.display import display, HTML
PAGE = URL + ('/live' if S3_OK else '/video')
display(HTML(f'<h3><a href="{PAGE}" target="_blank">{PAGE}</a></h3>'))
display(qrcode.make(PAGE).resize((320, 320)))
print('Autres pages :', URL + '/video', '|', URL + '/sessions')
if S3_OK:
    print('Sur la page : choisir Micro, Lien (direct YouTube ou flux HLS) ou Fichier.')
    print('Plusieurs écrans : une fois la session lancée, « Copier le lien » donne /live?sid=... pour suivre la même session.')

## 8. Facultatif : vérifier le serveur avant une démonstration

Contrôle ce serveur avec de la vraie parole arabe (FLEURS) : direct par WebSocket, rapport de fin de session, lecture vocale. Bilan PASS/FAIL avec les chiffres. Compter 3 à 4 minutes une fois les modèles en cache, 10 à 15 la première fois. Le rapport décharge le moteur du direct ; le script le recharge ensuite (environ une minute).

In [ ]:
if S3_OK:
    subprocess.run([sys.executable, 'verifier_reel.py', '--rapide', '--serveur', LOCAL,
                    '--sortie', '/content/verification_reelle.json'], cwd=APP)
else:
    print('Vérification sautée : elle teste le direct, qui demande S3.')

## 9. Facultatif : démarrer un direct depuis le notebook

Exemple avec le direct YouTube d'Al Jazeera. YouTube refuse parfois les serveurs de Colab : utiliser alors un fichier sur la page `/live`.

In [ ]:
# r = requests.post(LOCAL + '/api/live/url', json={'url': 'https://www.youtube.com/live/bNyUyrR0PHo', 'titre': 'Al Jazeera'}).json(); print(r)
# print(URL + '/live?sid=' + r['sid'])                                   # page à ouvrir pour suivre la session
# requests.post(LOCAL + f"/api/live/{r['sid']}/arreter").json()        # fin de session : bilan (retard, réécriture)

## 10. Mettre à jour sans changer de lien

Récupère la dernière version de la branche `integration` et relance le serveur. Le tunnel reste ouvert : le lien public ne change pas. Les sessions déjà enregistrées dans `/content/sessions` sont conservées.

In [ ]:
arreter_serveur()
subprocess.run(['git', '-C', DEPOT, 'pull', '-q'], check=True)
print('commit :', subprocess.run(['git', '-C', DEPOT, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
lancer_serveur()
if S3_OK:
    precharger_direct()
print('Serveur relancé, même lien :', URL)

## 11. Journal du serveur

In [ ]:
!tail -n 40 /content/server.log